# Caso 3 — IronMarch: radicalización y red social

**Dataset**: IronMarch (dump 2019, foro activo 2011–2017)  
**Idioma del foro**: mayoritariamente inglés, con presencia rusa  
**Tamaño**: ~1.200 usuarios, ~195k posts públicos, ~21k mensajes privados

**Narrativa**: IronMarch fue el principal foro neonazi aceleracionista en habla inglesa. Desmantelado en 2017, su base de datos se filtró en 2019. Lo que hace este dataset excepcional es que varios miembros son **públicamente identificados** y vinculados a ataques terroristas reales — lo que permite validar el análisis computacional con ground truth conocida.

> **Nota ética**: los datos son de dominio público en investigaciones periodísticas y judiciales. El análisis es estrictamente forense y académico.

## Estructura del notebook
1. [Setup](#s0)
2. [Carga y reconocimiento del schema](#s1)
3. [EDA — análisis exploratorio](#s2)
4. [Red social pública](#s3)
5. [Red de mensajes privados](#s4)
6. [Experimento: embeddings completo vs. muestreado](#s5)
7. [Topic modeling](#s6)
8. [NER — reconocimiento de entidades](#s7)
9. [Estilometría y atribución](#s8)
10. [Conclusiones](#s9)

<a id='s0'></a>
## Sección 0 — Setup

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import json
import re

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import networkx as nx
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

from src.utils import load_forum, load_or_compute, RESULTS_DIR, DATA_DIR
from src.embeddings import embed_users, compute_actor_centroids
from src.stylometry import compare_users, extract_features
from src.timezone import build_user_timezone_profile

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Datos:      {DATA_DIR}")
print(f"Resultados: {IM_RESULTS}")

In [ ]:
# --- Helpers para cacheo con timestamp ---
# Convención de nombres: {seccion}_{descripcion}_{YYYYMMDD_HHMMSS}.npz
# Ejemplo: s5a_embed_users_20260628_143022.npz

def save_timestamped(data: dict, base_dir: Path, section: str, name: str) -> Path:
    """Guarda un .npz con sección y timestamp en el nombre.
    
    section: identificador de sección, ej. 's5a', 's5b', 's5c'
    name:    descripción corta, ej. 'embed_users', 'centroids_full'
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = base_dir / f"{section}_{name}_{ts}.npz"
    np.savez_compressed(path, **data)
    print(f"Guardado: {path.name}")
    return path

def load_latest(base_dir: Path, pattern: str) -> dict | None:
    """Carga el .npz más reciente que matchea el patrón glob."""
    matches = sorted(base_dir.glob(pattern))
    if not matches:
        return None
    path = matches[-1]
    print(f"Cargando: {path.name}")
    return dict(np.load(path, allow_pickle=True))

print("Helpers de cacheo listos.")
print("Convención: {sección}_{descripción}_{YYYYMMDD_HHMMSS}.npz")

<a id='s1'></a>
## Sección 1 — Carga y reconocimiento del schema

Antes de asumir nada, inspeccionamos qué hay realmente en el dump.

In [ ]:
# Localizar el archivo de IronMarch
far_right_dir = DATA_DIR / 'Far Right Forum'
candidates = list(far_right_dir.glob('*IronMarch*.zip')) + list(far_right_dir.glob('*iron*.zip'))
candidates = [p for p in candidates if not p.name.startswith('._')]

if candidates:
    ironmarch_path = candidates[0]
    print(f"Dataset: {ironmarch_path.name}")
    print(f"Tamaño:  {ironmarch_path.stat().st_size / 1e6:.0f} MB")
else:
    raise FileNotFoundError(f"IronMarch zip no encontrado en {far_right_dir}")

In [ ]:
# Cargar el dump — el parser detecta automáticamente el formato IPS
raw = load_forum(ironmarch_path)

print(f"Tablas disponibles ({len(raw)}):")
for name, df in sorted(raw.items()):
    print(f"  {name:12s}: {len(df):>6,} filas | columnas: {list(df.columns)}")

In [ ]:
# Inspeccionar las tablas clave en crudo — sin asumir nada
for table in ['user', 'post', 'thread', 'forum', 'private_message']:
    if table in raw:
        print(f"\n=== {table} ===")
        print(raw[table].head(3).to_string())
        print(f"Columnas: {list(raw[table].columns)}")

In [ ]:
# Extraer tablas principales con nombres normalizados
users    = raw.get('user', pd.DataFrame())
posts    = raw.get('post', pd.DataFrame())
threads  = raw.get('thread', pd.DataFrame())
forums   = raw.get('forum', pd.DataFrame())
pms      = raw.get('private_message', pd.DataFrame())

# userid debe ser numérico — filas con contenido de texto en userid son errores del parser
# (ocurre por INSERTs multi-línea mal alineados)
for df in [posts, threads, pms]:
    for col in ['userid', 'sender_id', 'receiver_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.dropna(subset=[col], inplace=True)
            df[col] = df[col].astype(int).astype(str)

for col in ['userid']:
    if col in users.columns:
        users[col] = pd.to_numeric(users[col], errors='coerce')
        users.dropna(subset=[col], inplace=True)
        users[col] = users[col].astype(int).astype(str)

# Asegurar datetimes con timezone UTC
for df, col in [(posts, 'dateline'), (users, 'joindate'), (threads, 'dateline')]:
    if col in df.columns and len(df) > 0:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

print(f"Usuarios:          {len(users):,}")
print(f"Posts:             {len(posts):,}")
print(f"Hilos:             {len(threads):,}")
print(f"Subforos:          {len(forums):,}")
print(f"Mensajes privados: {len(pms):,}")

if 'dateline' in posts.columns:
    valid = posts['dateline'].dropna()
    if len(valid):
        print(f"\nRango temporal: {valid.min().strftime('%Y-%m-%d')} → {valid.max().strftime('%Y-%m-%d')}")

<a id='s2'></a>
## Sección 2 — EDA: análisis exploratorio

Primera pregunta: ¿qué tenemos realmente aquí? Antes de cualquier modelo, hay que entender el dato.

In [ ]:
# Distribución de posts por usuario (power-law)
# En casi todos los foros: pocos usuarios escriben muchísimo, la mayoría casi nada
if 'userid' in posts.columns:
    post_counts = posts.groupby('userid').size().sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(post_counts.values, bins=60, color='#E94E4E', alpha=0.85)
    axes[0].set_title('Distribución de posts por usuario')
    axes[0].set_xlabel('Posts')
    axes[0].set_ylabel('Usuarios')
    
    # Log-log revela la power-law
    bins = np.logspace(0, np.log10(post_counts.max() + 1), 40)
    axes[1].hist(post_counts.values, bins=bins, color='#E94E4E', alpha=0.85)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('Power-law (escala log-log)')
    axes[1].set_xlabel('Posts (log)')
    axes[1].set_ylabel('Usuarios (log)')
    
    plt.suptitle('IronMarch — actividad de usuarios', y=1.01)
    plt.tight_layout()
    plt.show()
    
    # Estadísticas clave
    print(f"Usuarios con 1 post:      {(post_counts == 1).sum():,} ({(post_counts == 1).mean()*100:.1f}%)")
    print(f"Usuarios con ≥10 posts:   {(post_counts >= 10).sum():,} ({(post_counts >= 10).mean()*100:.1f}%)")
    print(f"Usuarios con ≥100 posts:  {(post_counts >= 100).sum():,}")
    print(f"Top 10% de usuarios generan {post_counts.nlargest(int(len(post_counts)*0.1)).sum() / post_counts.sum()*100:.0f}% de los posts")

In [ ]:
# Top 20 usuarios más prolíficos
if 'userid' in posts.columns and 'username' in users.columns:
    uid_to_name = dict(zip(users['userid'], users['username']))
    top_users = (
        posts.groupby('userid').size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
        .rename(columns={0: 'posts', 'userid': 'userid'})
    )
    top_users.columns = ['userid', 'posts']
    top_users['username'] = top_users['userid'].map(uid_to_name)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top_users['username'].fillna(top_users['userid'].astype(str))[::-1],
                   top_users['posts'][::-1], color='#E94E4E', alpha=0.85)
    ax.set_xlabel('Posts')
    ax.set_title('Top 20 usuarios más activos — IronMarch')
    plt.tight_layout()
    plt.show()

In [ ]:
# Evolución temporal con eventos del mundo real
EVENTS = {
    '2011-07': 'Breivik (Utøya)',
    '2012-08': 'Wisconsin Sikh Temple',
    '2015-06': 'Charleston (Emanuel AME)',
    '2015-11': 'Atentados París',
    '2016-11': 'Elección Trump',
    '2017-08': 'Charlottesville',
    '2017-09': 'IronMarch cerrado',
}

if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline'])
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]
    monthly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('M')).size()
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(monthly)), monthly.values, color='#E94E4E', linewidth=1.5)
    ax.fill_between(range(len(monthly)), monthly.values, alpha=0.2, color='#E94E4E')
    
    month_index = {str(p): i for i, p in enumerate(monthly.index)}
    for event_ym, label in EVENTS.items():
        if event_ym in month_index:
            idx = month_index[event_ym]
            ax.axvline(idx, color='#FFD700', alpha=0.7, linestyle='--', linewidth=1)
            ax.text(idx + 0.2, monthly.max() * 0.92, label,
                    rotation=90, fontsize=7, color='#FFD700', va='top')
    
    tick_step = max(1, len(monthly) // 14)
    ax.set_xticks(range(0, len(monthly), tick_step))
    ax.set_xticklabels([str(monthly.index[i]) for i in range(0, len(monthly), tick_step)],
                       rotation=45, fontsize=8)
    ax.set_title('IronMarch — posts mensuales con eventos externos')
    ax.set_ylabel('Posts / mes')
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap: actividad por hora UTC y día de semana → inferir timezone dominante
if 'dateline' in posts.columns:
    df_tz = posts.dropna(subset=['dateline']).copy()
    df_tz['hour']    = df_tz['dateline'].dt.hour
    df_tz['weekday'] = df_tz['dateline'].dt.weekday
    
    heatmap_data = df_tz.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
    day_names = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    heatmap_data.index = [day_names[i] for i in heatmap_data.index]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.1,
                cbar_kws={'label': 'Posts'})
    ax.set_title('IronMarch — actividad por hora UTC y día de la semana')
    ax.set_xlabel('Hora UTC')
    plt.tight_layout()
    plt.show()
    
    tz_profile = build_user_timezone_profile(df_tz)
    print("\nTop UTC offsets inferidos (timezone probable de los usuarios):")
    print(tz_profile['inferred_utc_offset'].value_counts().head(8).to_string())

In [ ]:
# Subforos más activos
# posts solo tiene threadid — hay que joinear con threads para obtener forumid
if len(threads) > 0 and 'forumid' in threads.columns and 'threadid' in posts.columns:
    thread_to_forum = threads.set_index('threadid')['forumid'].to_dict()
    posts_with_forum = posts.copy()
    posts_with_forum['forumid'] = posts_with_forum['threadid'].map(thread_to_forum)

    if len(forums) > 0:
        fname_col = next((c for c in forums.columns if c == 'name'), None)
        fid_col   = next((c for c in forums.columns if c == 'forumid'), None)
        if fname_col and fid_col:
            fid_to_name = dict(zip(forums[fid_col].astype(str), forums[fname_col]))
            posts_with_forum['forumid'] = posts_with_forum['forumid'].astype(str)

            forum_counts = (
                posts_with_forum.groupby('forumid').size()
                .sort_values(ascending=False)
                .head(20)
                .reset_index()
            )
            forum_counts.columns = ['forumid', 'posts']
            forum_counts['name'] = forum_counts['forumid'].map(fid_to_name).fillna('(desconocido)')

            fig, ax = plt.subplots(figsize=(10, 8))
            ax.barh(forum_counts['name'][::-1], forum_counts['posts'][::-1],
                    color='#E94E4E', alpha=0.85)
            ax.set_xlabel('Posts')
            ax.set_title('Top 20 subforos más activos — IronMarch')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Columnas de forums: {list(forums.columns)}")
    else:
        print("Sin tabla de foros cargada.")
else:
    print(f"Columnas de posts: {list(posts.columns)}")
    print(f"Columnas de threads: {list(threads.columns) if len(threads) > 0 else 'vacío'}")

In [ ]:
# Detección de idioma en muestra de posts
try:
    from langdetect import detect, LangDetectException
    
    text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
    sample = posts.dropna(subset=[text_col]).sample(min(1000, len(posts)), random_state=42)
    
    def safe_detect(text):
        try:
            return detect(str(text)[:500])
        except LangDetectException:
            return 'unknown'
    
    sample = sample.copy()
    sample['lang'] = sample[text_col].map(safe_detect)
    lang_counts = sample['lang'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(lang_counts.index, lang_counts.values, color='#E94E4E', alpha=0.85)
    ax.set_title('Idiomas detectados (muestra de 1.000 posts)')
    ax.set_ylabel('Posts')
    plt.tight_layout()
    plt.show()
    
    print(lang_counts.to_string())
except ImportError:
    print("langdetect no instalado.")

<a id='s3'></a>
## Sección 3 — Red social pública

Grafo de co-participación en hilos: dos usuarios están conectados si escribieron en el mismo hilo. El **betweenness centrality** identifica los brokers — usuarios que conectan subcomunidades distintas dentro del foro.

In [ ]:
# Construir grafo de co-participación en hilos
G_public = nx.Graph()

thread_col = 'threadid' if 'threadid' in posts.columns else None

if thread_col and 'userid' in posts.columns:
    # Forzar str en ambas columnas para evitar comparaciones tipo-mixto
    df_graph = posts[['threadid', 'userid']].copy()
    df_graph['threadid'] = df_graph['threadid'].astype(str)
    df_graph['userid']   = df_graph['userid'].astype(str)
    df_graph = df_graph[df_graph['threadid'] != 'nan']

    thread_users = df_graph.groupby('threadid')['userid'].apply(list)

    edge_weights = Counter()
    for participants in thread_users:
        unique = list(set(participants))
        for i in range(len(unique)):
            for j in range(i + 1, min(i + 10, len(unique))):
                pair = tuple(sorted([unique[i], unique[j]]))
                edge_weights[pair] += 1

    for (u, v), w in edge_weights.items():
        G_public.add_edge(u, v, weight=w)

    print(f"Grafo público: {G_public.number_of_nodes():,} nodos, {G_public.number_of_edges():,} aristas")
else:
    print(f"Columnas disponibles en posts: {list(posts.columns)}")

In [ ]:
# Calcular betweenness centrality (con sampling para grafos grandes)
uid_to_name = dict(zip(users['userid'], users['username'])) if 'username' in users.columns else {}

if G_public.number_of_nodes() > 0:
    degree_cent  = nx.degree_centrality(G_public)
    
    k_sample = min(300, G_public.number_of_nodes())
    print(f"Calculando betweenness (k={k_sample})...")
    btw_cent = nx.betweenness_centrality(G_public, k=k_sample, normalized=True, weight='weight')
    
    top_brokers = sorted(btw_cent.items(), key=lambda x: x[1], reverse=True)[:15]
    
    print("\nTop 15 brokers (betweenness centrality):")
    broker_records = []
    for uid, score in top_brokers:
        name = uid_to_name.get(uid, str(uid))
        deg  = degree_cent.get(uid, 0)
        broker_records.append({'userid': uid, 'username': name,
                               'betweenness': round(score, 5), 'degree': round(deg, 4)})
        print(f"  {name:<25} btw={score:.4f}  deg={deg:.4f}")
    
    brokers_df = pd.DataFrame(broker_records)
else:
    print("Grafo vacío — revisar carga de datos.")

In [ ]:
# Visualización interactiva con Plotly — zoom, hover, pannable
# Solo top 150 nodos por betweenness para evitar solapamiento
import plotly.graph_objects as go

if G_public.number_of_nodes() > 0:
    largest_cc = max(nx.connected_components(G_public), key=len)
    G_main = G_public.subgraph(largest_cc).copy()

    # Seleccionar top 150 por betweenness dentro del componente principal
    top_nodes = sorted(
        [n for n in G_main.nodes() if n in btw_cent],
        key=lambda n: btw_cent[n], reverse=True
    )[:150]
    G_viz = G_main.subgraph(top_nodes).copy()

    pos = nx.spring_layout(G_viz, k=1.2, iterations=80, seed=42)

    # Aristas
    edge_x, edge_y = [], []
    for u, v in G_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.4, color='rgba(150,150,150,0.3)'),
        hoverinfo='none',
    )

    # Nodos
    node_x     = [pos[n][0] for n in G_viz.nodes()]
    node_y     = [pos[n][1] for n in G_viz.nodes()]
    node_btw   = [btw_cent.get(n, 0) for n in G_viz.nodes()]
    node_deg   = [G_viz.degree(n) for n in G_viz.nodes()]
    node_names = [uid_to_name.get(n, f'uid:{n}') for n in G_viz.nodes()]
    node_sizes = [b * 60 + 6 for b in node_btw]

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_btw,
            colorscale='YlOrRd',
            colorbar=dict(title='Betweenness'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if btw_cent.get(n, 0) > sorted(node_btw, reverse=True)[min(12, len(node_btw)-1)] else ''
              for n, name in zip(G_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[f'{name}<br>btw={btw_cent.get(n,0):.4f}<br>conexiones={G_viz.degree(n)}'
                   for n, name in zip(G_viz.nodes(), node_names)],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red IronMarch — top 150 nodos (hover para detalles, zoom libre)',
            template='plotly_dark',
            showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print("Grafo vacío.")

<a id='s4'></a>
## Sección 4 — Red de mensajes privados

21k mensajes privados entre miembros — dato infrecuente en leaks. La red de PMs revela quién habla con quién **fuera del espacio público** del foro. La pregunta clave: ¿los brokers públicos son los mismos que los privados?

In [ ]:
# Inspeccionar schema de mensajes privados
if len(pms) > 0:
    print(f"Columnas de PMs: {list(pms.columns)}")
    print(f"\nPrimeras filas:")
    print(pms.head(3).to_string())
else:
    print("Sin datos de PMs. Columnas disponibles en raw:", list(raw.keys()))

In [ ]:
# Construir grafo de mensajes privados
G_pm = nx.DiGraph()

# Identificar columnas de sender/receiver según schema real
sender_col   = next((c for c in pms.columns if 'from' in c.lower() or 'author' in c.lower() or 'sender' in c.lower()), None)
receiver_col = next((c for c in pms.columns if 'to' in c.lower() or 'recipient' in c.lower() or 'receiver' in c.lower()), None)

print(f"Columna sender:   {sender_col}")
print(f"Columna receiver: {receiver_col}")

if sender_col and receiver_col:
    pm_edges = pms[[sender_col, receiver_col]].dropna()
    pm_edges = pm_edges[pm_edges[sender_col] != pm_edges[receiver_col]]
    
    for _, row in pm_edges.iterrows():
        s, r = row[sender_col], row[receiver_col]
        if G_pm.has_edge(s, r):
            G_pm[s][r]['weight'] += 1
        else:
            G_pm.add_edge(s, r, weight=1)
    
    print(f"\nGrafo PMs: {G_pm.number_of_nodes():,} nodos, {G_pm.number_of_edges():,} aristas")
else:
    print(f"\nColumnas disponibles en PMs: {list(pms.columns)}")
    print("Ajustar sender_col y receiver_col manualmente.")

In [ ]:
# Comparar brokers públicos vs. privados
if G_pm.number_of_nodes() > 0:
    G_pm_und = G_pm.to_undirected()
    k_pm = min(200, G_pm_und.number_of_nodes())
    btw_pm = nx.betweenness_centrality(G_pm_und, k=k_pm, normalized=True)
    
    top_pm_brokers = sorted(btw_pm.items(), key=lambda x: x[1], reverse=True)[:15]
    
    print("Top 15 brokers en red de PMs:")
    for uid, score in top_pm_brokers:
        name = uid_to_name.get(uid, str(uid))
        print(f"  {name:<25} btw={score:.4f}")
    
    # Intersección con brokers públicos
    top_public_uids = {uid for uid, _ in top_brokers[:15]}
    top_pm_uids     = {uid for uid, _ in top_pm_brokers[:15]}
    overlap = top_public_uids & top_pm_uids
    
    print(f"\nIntersección (brokers en ambas redes): {len(overlap)} usuarios")
    for uid in overlap:
        print(f"  {uid_to_name.get(uid, str(uid))}")
else:
    print("Sin datos de PM para construir la red.")

In [ ]:
# Visualización interactiva de la red de PMs con Plotly
import plotly.graph_objects as go

if G_pm.number_of_nodes() > 0:
    top_pm_nodes = sorted(G_pm_und.degree(), key=lambda x: x[1], reverse=True)[:150]
    top_pm_set   = {n for n, _ in top_pm_nodes}
    G_pm_viz     = G_pm_und.subgraph(top_pm_set).copy()

    pos = nx.spring_layout(G_pm_viz, k=0.8, iterations=60, seed=42)

    # Aristas
    edge_x, edge_y = [], []
    for u, v in G_pm_viz.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.5, color='rgba(150,150,150,0.25)'),
        hoverinfo='none',
    )

    # Nodos
    node_x     = [pos[n][0] for n in G_pm_viz.nodes()]
    node_y     = [pos[n][1] for n in G_pm_viz.nodes()]
    node_btw   = [btw_pm.get(n, 0) for n in G_pm_viz.nodes()]
    node_names = [uid_to_name.get(n, f'uid:{n}') for n in G_pm_viz.nodes()]
    node_sizes = [b * 80 + 6 for b in node_btw]

    top8_pm = {uid for uid, _ in top_pm_brokers[:8]}

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=node_sizes,
            color=node_btw,
            colorscale='Plasma',
            colorbar=dict(title='Betweenness (PMs)'),
            line=dict(width=0.5, color='white'),
        ),
        text=[name if n in top8_pm else ''
              for n, name in zip(G_pm_viz.nodes(), node_names)],
        textposition='top center',
        textfont=dict(size=9, color='white'),
        hovertext=[
            f'{name}<br>btw={btw_pm.get(n, 0):.4f}<br>conexiones PM={G_pm_viz.degree(n)}'
            for n, name in zip(G_pm_viz.nodes(), node_names)
        ],
        hoverinfo='text',
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title='Red de mensajes privados — IronMarch (top 150, hover para detalles)',
            template='plotly_dark',
            showlegend=False,
            width=1000, height=750,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=50, b=20),
        )
    )
    fig.show()
else:
    print("Grafo de PMs vacío.")

<a id='s5'></a>
## Sección 5 — Experimento metodológico: embeddings

Tres enfoques distintos para representar usuarios como vectores. El objetivo es entender qué se pierde (o no) al muestrear.

| Parte | Método | Tiempo estimado | Guardado con timestamp |
|---|---|---|---|
| A | `embed_users` — concatenar todos los posts por usuario → 1 vector | ~12 min | ✓ |
| B | `compute_actor_centroids` **completo** — 1 embedding por post, promedio por usuario | ~6–10 h | ✓ |
| C | `compute_actor_centroids` **muestreado** — top 50 posts por usuario | ~1.5 h | ✓ |
| D | Comparativa de rankings de similitud entre A, B y C | instantáneo | — |

> Las partes B y C pueden correr el fin de semana. El notebook carga el resultado más reciente automáticamente.

In [ ]:
# Preparar DataFrame de posts para embeddings
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

posts_embed = posts[['userid', text_col]].copy()
posts_embed = posts_embed.dropna(subset=[text_col])
posts_embed = posts_embed[posts_embed[text_col].astype(str).str.strip().str.len() > 20]
posts_embed = posts_embed.rename(columns={text_col: 'pagetext'})

post_counts = posts_embed.groupby('userid').size()
print(f"Posts con texto válido: {len(posts_embed):,}")
print(f"Usuarios con ≥5 posts:  {(post_counts >= 5).sum():,}")
print(f"Usuarios con ≥10 posts: {(post_counts >= 10).sum():,}")

### Parte A — `embed_users`: concatenación completa (~12 min)

In [ ]:
# Parte A: embed_users — concatena todos los posts por usuario → 1 vector por usuario
# Tiempo estimado: ~12 min | Guardado: s5a_embed_users_{timestamp}.npz
cached_a = load_latest(IM_RESULTS, 's5a_embed_users_*.npz')

if cached_a is not None:
    user_ids_a = cached_a['user_ids'].tolist()
    vectors_a  = cached_a['vectors']
    print(f"Parte A cargada: {len(user_ids_a):,} usuarios, dim={vectors_a.shape[1]}")
else:
    print("Computando Parte A (embed_users)...")
    user_ids_a, vectors_a = embed_users(posts_embed, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(user_ids_a, dtype='U64'), 'vectors': vectors_a},
        IM_RESULTS, 's5a', 'embed_users'
    )
    print(f"Parte A completada: {len(user_ids_a):,} usuarios")

### Parte B — `compute_actor_centroids` completo (~6–10 h)

> ⚠️ Esta celda puede tardar horas. Ejecutar y dejar correr. El resultado se guarda con timestamp.

In [ ]:
# Parte B: centroids completo — 1 embedding por post, promedio por usuario
# Tiempo estimado: ~6–10 h | Guardado: s5b_centroids_full_{timestamp}.npz
# ⚠️ Dejar correr el fin de semana
cached_b = load_latest(IM_RESULTS, 's5b_centroids_full_*.npz')

if cached_b is not None:
    user_ids_b = cached_b['user_ids'].tolist()
    vectors_b  = cached_b['vectors']
    print(f"Parte B cargada: {len(user_ids_b):,} usuarios, dim={vectors_b.shape[1]}")
else:
    print(f"Computando Parte B — todos los posts ({len(posts_embed):,})...")
    print("Estimado: 6–10 horas. Dejar corriendo.")
    user_ids_b, vectors_b = compute_actor_centroids(posts_embed, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(user_ids_b, dtype='U64'), 'vectors': vectors_b},
        IM_RESULTS, 's5b', 'centroids_full'
    )
    print(f"Parte B completada: {len(user_ids_b):,} usuarios")

### Parte C — `compute_actor_centroids` muestreado (~1.5 h)

Top 50 posts por usuario — muestra estratificada, no aleatoria.

In [ ]:
# Construir muestra estratificada: top 50 posts por usuario (los más largos)
MAX_POSTS_PER_USER = 50

posts_sampled = (
    posts_embed
    .assign(text_len=posts_embed['pagetext'].str.len())
    .sort_values('text_len', ascending=False)
    .groupby('userid')
    .head(MAX_POSTS_PER_USER)
    .drop(columns='text_len')
    .reset_index(drop=True)
)

print(f"Posts en muestra:  {len(posts_sampled):,} ({len(posts_sampled)/len(posts_embed)*100:.1f}% del total)")
print(f"Usuarios cubiertos: {posts_sampled['userid'].nunique():,}")

In [ ]:
# Parte C: centroids muestreado — top 50 posts/usuario
# Tiempo estimado: ~1.5 h | Guardado: s5c_centroids_sampled_{timestamp}.npz
cached_c = load_latest(IM_RESULTS, 's5c_centroids_sampled_*.npz')

if cached_c is not None:
    user_ids_c = cached_c['user_ids'].tolist()
    vectors_c  = cached_c['vectors']
    print(f"Parte C cargada: {len(user_ids_c):,} usuarios, dim={vectors_c.shape[1]}")
else:
    print(f"Computando Parte C — muestra de {len(posts_sampled):,} posts...")
    user_ids_c, vectors_c = compute_actor_centroids(posts_sampled, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(user_ids_c, dtype='U64'), 'vectors': vectors_c},
        IM_RESULTS, 's5c', 'centroids_sampled'
    )
    print(f"Parte C completada: {len(user_ids_c):,} usuarios")

### Parte D — Comparativa de rankings

¿Los tres métodos coinciden en quiénes son los usuarios más similares entre sí?

In [ ]:
# Comparar rankings de similitud entre pares de métodos disponibles
# Usamos correlación de Spearman sobre las similitudes pairwise

def pairwise_similarities(user_ids: list, vectors: np.ndarray) -> pd.Series:
    """Similitudes coseno para todos los pares (triu), versión vectorizada."""
    sim_matrix = cosine_similarity(vectors)
    np.fill_diagonal(sim_matrix, 0)
    idx_i, idx_j = np.triu_indices(len(user_ids), k=1)
    keys = pd.MultiIndex.from_arrays([
        [str(user_ids[i]) for i in idx_i],
        [str(user_ids[j]) for j in idx_j],
    ])
    return pd.Series(sim_matrix[idx_i, idx_j], index=keys)

def compare_rankings(sims_x: pd.Series, sims_y: pd.Series, label_x: str, label_y: str) -> float:
    """Correlación de Spearman entre dos distribuciones de similitudes para los pares comunes."""
    common = sims_x.index.intersection(sims_y.index)
    if len(common) == 0:
        print(f"Sin pares comunes entre {label_x} y {label_y}.")
        return 0.0
    rho, p = spearmanr(sims_x[common], sims_y[common])
    print(f"{label_x} vs {label_y}: Spearman ρ={rho:.4f} (p={p:.2e}) sobre {len(common):,} pares")
    return rho

# Calcular similitudes para cada método disponible
available = {}
if 'vectors_a' in dir() and len(vectors_a) > 1:
    print("Calculando pares para A...")
    available['embed_users (A)'] = pairwise_similarities(user_ids_a, vectors_a)
if 'vectors_b' in dir() and len(vectors_b) > 1:
    print("Calculando pares para B...")
    available['centroids_full (B)'] = pairwise_similarities(user_ids_b, vectors_b)
if 'vectors_c' in dir() and len(vectors_c) > 1:
    print("Calculando pares para C...")
    available['centroids_sampled (C)'] = pairwise_similarities(user_ids_c, vectors_c)

print("\nCorrelaciones de Spearman entre rankings de similitud:")
spearman_results = {}
keys = list(available.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        rho = compare_rankings(available[keys[i]], available[keys[j]], keys[i], keys[j])
        spearman_results[(keys[i], keys[j])] = rho

In [ ]:
# Scatter de similitudes pairwise — visual de la correlación entre métodos
# Cada punto es un par de usuarios; el eje X es la similitud según método 1,
# el eje Y según método 2. Si están alineados, los métodos coinciden en rankings.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(available) >= 2:
    pairs_to_plot = [(keys[i], keys[j]) for i in range(len(keys)) for j in range(i+1, len(keys))]
    fig = make_subplots(
        rows=1, cols=len(pairs_to_plot),
        subplot_titles=[f'{x.split("(")[1].rstrip(")")} vs {y.split("(")[1].rstrip(")")}' for x, y in pairs_to_plot],
    )

    rng = np.random.default_rng(42)
    for col_idx, (lx, ly) in enumerate(pairs_to_plot, 1):
        common = available[lx].index.intersection(available[ly].index)
        sx = available[lx][common].values
        sy = available[ly][common].values

        # Muestra aleatoria para no saturar (max 5000 puntos)
        n_show = min(5000, len(common))
        idx_s = rng.choice(len(common), n_show, replace=False)

        rho = spearman_results.get((lx, ly), spearman_results.get((ly, lx), float('nan')))

        fig.add_trace(
            go.Scatter(
                x=sx[idx_s], y=sy[idx_s],
                mode='markers',
                marker=dict(size=2, opacity=0.25, color='#E94E4E'),
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        # Línea diagonal de referencia (correlación perfecta)
        fig.add_trace(
            go.Scatter(
                x=[0, 1], y=[0, 1],
                mode='lines',
                line=dict(color='rgba(255,255,255,0.3)', dash='dash'),
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        label_x = lx.split('(')[0].strip()
        label_y = ly.split('(')[0].strip()
        fig.update_xaxes(title_text=label_x, range=[0, 1], row=1, col=col_idx)
        fig.update_yaxes(title_text=label_y, range=[0, 1], row=1, col=col_idx)
        fig.layout.annotations[col_idx - 1].text += f'<br>ρ={rho:.3f}'

    fig.update_layout(
        title='Similitudes pairwise — correlación entre métodos de embedding',
        template='plotly_dark',
        height=450, width=400 * len(pairs_to_plot),
    )
    fig.show()
else:
    print("Se necesitan al menos 2 métodos para la comparativa visual.")

In [ ]:
# UMAP comparativo — los tres métodos en subplots
# Permite ver si la estructura de clusters cambia según cómo representamos a los usuarios
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import umap as umap_lib
import hdbscan

method_data = {}
if 'vectors_a' in dir() and len(vectors_a) > 1:
    method_data['A: embed_users'] = (user_ids_a, vectors_a)
if 'vectors_b' in dir() and len(vectors_b) > 1:
    method_data['B: centroids_full'] = (user_ids_b, vectors_b)
if 'vectors_c' in dir() and len(vectors_c) > 1:
    method_data['C: centroids_sampled'] = (user_ids_c, vectors_c)

if not method_data:
    print("Sin embeddings disponibles. Ejecutar primero las partes A, B o C.")
else:
    n_methods = len(method_data)
    fig = make_subplots(
        rows=1, cols=n_methods,
        subplot_titles=list(method_data.keys()),
        horizontal_spacing=0.05,
    )

    for col_idx, (label, (ids, vecs)) in enumerate(method_data.items(), 1):
        print(f"UMAP {label}...", end=' ', flush=True)
        reducer  = umap_lib.UMAP(n_components=2, n_neighbors=min(15, len(vecs)-1), random_state=42)
        coords   = reducer.fit_transform(vecs)

        clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
        clusters  = clusterer.fit_predict(vecs)
        n_cl      = len(set(clusters)) - (1 if -1 in clusters else 0)
        print(f"{n_cl} clusters, {(clusters==-1).sum()} ruido")

        names = [uid_to_name.get(uid, str(uid)) for uid in ids]

        fig.add_trace(
            go.Scatter(
                x=coords[:, 0], y=coords[:, 1],
                mode='markers',
                marker=dict(
                    size=5,
                    color=clusters,
                    colorscale='Plasma',
                    showscale=False,
                    opacity=0.8,
                    line=dict(width=0, color='black'),
                ),
                text=[f'{n}<br>cluster={c}' for n, c in zip(names, clusters)],
                hoverinfo='text',
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col_idx)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col_idx)

    fig.update_layout(
        title='UMAP + HDBSCAN — comparativa de métodos de embedding (hover = usuario)',
        template='plotly_dark',
        height=520, width=420 * n_methods,
        margin=dict(l=10, r=10, t=80, b=10),
    )
    fig.show()

<a id='s6'></a>
## Sección 6 — Topic modeling

BERTopic descubre los temas principales del foro de forma no supervisada. Luego comparamos los topics automáticos con los subforos reales — si coinciden, el modelo es válido.

In [ ]:
# Preparar corpus de textos para topic modeling
# Filtramos posts cortos y los primeros 10.000 más largos para que sea manejable
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

corpus_df = posts[[text_col, 'userid'] + (['forumid'] if 'forumid' in posts.columns else [])].copy()
corpus_df = corpus_df.dropna(subset=[text_col])
corpus_df['text_clean'] = corpus_df[text_col].astype(str).str.replace(r'<[^>]+>', ' ', regex=True).str.strip()
corpus_df = corpus_df[corpus_df['text_clean'].str.len() > 100]

# Muestra representativa: hasta 15.000 posts
if len(corpus_df) > 15000:
    corpus_df = corpus_df.sample(15000, random_state=42)

documents = corpus_df['text_clean'].tolist()
print(f"Documentos para topic modeling: {len(documents):,}")

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Vectorizer con stopwords inglesas
vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model = BERTopic(
    vectorizer_model=vectorizer,
    min_topic_size=15,
    nr_topics='auto',
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(documents)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"\nTopics encontrados: {n_topics}")
print(f"Posts sin topic (-1): {topics.count(-1):,}")

In [ ]:
# Inspeccionar top topics — barchart interactivo de keywords por topic
import plotly.graph_objects as go
from plotly.subplots import make_subplots

topic_info = topic_model.get_topics()
valid_topics = {t: words for t, words in topic_info.items() if t != -1}

# Mostrar top N topics por volumen
topic_sizes = {t: topics.count(t) for t in valid_topics}
top_topic_ids = sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:12]

n_cols = 3
n_rows = (len(top_topic_ids) + n_cols - 1) // n_cols
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'Topic {t} ({topic_sizes[t]} posts)' for t in top_topic_ids],
    vertical_spacing=0.12, horizontal_spacing=0.08,
)

for idx, t in enumerate(top_topic_ids):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    words = [w for w, _ in valid_topics[t][:8]]
    scores = [s for _, s in valid_topics[t][:8]]
    fig.add_trace(
        go.Bar(
            x=scores[::-1], y=words[::-1],
            orientation='h',
            marker_color='#E94E4E',
            showlegend=False,
        ),
        row=row, col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.update_layout(
    title=f'Top 12 topics — IronMarch ({n_topics} topics, {topics.count(-1):,} posts sin topic)',
    template='plotly_dark',
    height=280 * n_rows, width=1100,
    margin=dict(l=10, r=10, t=80, b=10),
)
fig.show()

In [ ]:
# Validación: heatmap topic vs. subforo (plotly)
# Si el topic modeling capta la estructura del foro, los topics deberían concentrarse en subforos concretos
import plotly.graph_objects as go

if 'forumid' in corpus_df.columns and len(forums) > 0:
    fname_col = next((c for c in forums.columns if 'name' in c.lower() or 'title' in c.lower()), None)
    fid_col   = next((c for c in forums.columns if 'id' in c.lower() and 'forum' in c.lower()), None)

    if fname_col and fid_col:
        fid_to_name = dict(zip(forums[fid_col].astype(str), forums[fname_col]))
        corpus_df = corpus_df.copy()
        corpus_df['topic'] = topics
        corpus_df['subforum'] = corpus_df['forumid'].astype(str).map(fid_to_name).fillna('(otros)')

        # Cross-tab: topic × subforo
        ct = pd.crosstab(corpus_df['topic'], corpus_df['subforum'])
        ct = ct.loc[ct.index != -1]  # quitar outliers
        # normalizar por fila para ver la distribución dentro de cada topic
        ct_norm = ct.div(ct.sum(axis=1), axis=0)

        # Mostrar solo top 10 subforos por volumen y top 15 topics por tamaño
        top_subforums = ct.sum().nlargest(10).index.tolist()
        top_topics    = ct.sum(axis=1).nlargest(15).index.tolist()
        heat = ct_norm.loc[top_topics, top_subforums]

        # Labels de topics: "T{id}: kw1, kw2, kw3"
        def topic_label(t):
            kws = [w for w, _ in topic_model.get_topic(t)[:3]]
            return f'T{t}: {", ".join(kws)}'

        row_labels = [topic_label(t) for t in heat.index]

        fig = go.Figure(go.Heatmap(
            z=heat.values,
            x=[sf[:30] for sf in heat.columns],
            y=row_labels,
            colorscale='YlOrRd',
            hovertemplate='Topic: %{y}<br>Subforo: %{x}<br>Proporción: %{z:.2f}<extra></extra>',
        ))
        fig.update_layout(
            title='Distribución de topics por subforo (proporción por fila)',
            template='plotly_dark',
            height=max(400, 30 * len(top_topics) + 100),
            width=1000,
            xaxis=dict(tickangle=-35),
            margin=dict(l=200, r=20, t=60, b=120),
        )
        fig.show()

        # Interpretación automática: subforo dominante por topic
        print("\nSubforo dominante por topic:")
        for t in top_topics:
            dominant = ct.loc[t].idxmax() if t in ct.index else '?'
            kws = [w for w, _ in topic_model.get_topic(t)[:4]]
            print(f"  T{t:<3} ({', '.join(kws):<35}) → {dominant}")
    else:
        print(f"Columnas de forums disponibles: {list(forums.columns)}")
else:
    print("Sin columna forumid disponible para validación.")

In [ ]:
# Usuarios más activos por topic — ¿quién lidera cada discurso?
if 'topic' in corpus_df.columns and 'userid' in corpus_df.columns:
    user_topic = corpus_df[corpus_df['topic'] != -1].groupby(['topic', 'userid']).size().reset_index(name='posts')
    
    top_topic_ids_limited = sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:8]
    
    rows_data = []
    for t in top_topic_ids_limited:
        kws = [w for w, _ in topic_model.get_topic(t)[:3]]
        top_in_topic = (
            user_topic[user_topic['topic'] == t]
            .sort_values('posts', ascending=False)
            .head(5)
        )
        for _, row in top_in_topic.iterrows():
            uname = uid_to_name.get(str(row['userid']), str(row['userid']))
            rows_data.append({
                'topic': f'T{t}: {", ".join(kws)}',
                'usuario': uname,
                'posts_en_topic': int(row['posts']),
            })
    
    df_ut = pd.DataFrame(rows_data)
    
    import plotly.express as px
    fig = px.bar(
        df_ut, x='posts_en_topic', y='usuario', color='topic',
        facet_col='topic', facet_col_wrap=4,
        orientation='h',
        template='plotly_dark',
        title='Top 5 usuarios por topic (top 8 topics)',
        height=600, width=1100,
    )
    fig.update_yaxes(matches=None)
    fig.update_layout(showlegend=False)
    fig.show()
else:
    print("Ejecutar primero la celda de validación topic-subforo.")

<a id='s7'></a>
## Sección 7 — NER: reconocimiento de entidades

Extracción de personas, organizaciones, eventos e ideologías mencionados en el foro con `qwen2.5:14b` local.

### Comparativa de estrategias de muestreo

¿La distribución de entidades es estable independientemente de cómo muestreamos?

| Estrategia | Descripción |
|---|---|
| Aleatoria | 500 posts al azar |
| Estratificada por subforo | Posts proporcionales al volumen de cada subforo |
| Top usuarios | 20 posts de los 25 usuarios más activos |

In [ ]:
# Construir las tres muestras
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
SAMPLE_SIZE = 500

posts_text = posts.dropna(subset=[text_col]).copy()
posts_text = posts_text[posts_text[text_col].astype(str).str.len() > 100]

# Muestra A: aleatoria
sample_random = posts_text.sample(min(SAMPLE_SIZE, len(posts_text)), random_state=42)

# Muestra B: estratificada por subforo
if 'forumid' in posts_text.columns:
    forum_sizes = posts_text['forumid'].value_counts(normalize=True)
    stratified = []
    for fid, prop in forum_sizes.items():
        n = max(1, int(SAMPLE_SIZE * prop))
        subset = posts_text[posts_text['forumid'] == fid].sample(min(n, len(posts_text[posts_text['forumid'] == fid])), random_state=42)
        stratified.append(subset)
    sample_stratified = pd.concat(stratified).head(SAMPLE_SIZE)
else:
    sample_stratified = sample_random  # fallback si no hay forumid

# Muestra C: top usuarios
top_user_ids = posts_text.groupby('userid').size().nlargest(25).index
sample_top_users = (
    posts_text[posts_text['userid'].isin(top_user_ids)]
    .groupby('userid')
    .head(20)
    .reset_index(drop=True)
)

print(f"Muestra aleatoria:          {len(sample_random):,} posts")
print(f"Muestra estratificada:      {len(sample_stratified):,} posts")
print(f"Muestra top usuarios:       {len(sample_top_users):,} posts")

In [ ]:
# Función de NER con qwen2.5:14b vía Ollama
import ollama

NER_SYSTEM = """You are a threat intelligence analyst. Extract named entities from forum posts.
Reply ONLY with valid JSON array: [{"entity": "...", "type": "..."}]
Allowed types: PERSON, ORGANIZATION, LOCATION, EVENT, IDEOLOGY
If no entities found, reply: []"""

def extract_entities(text: str, model: str = 'qwen2.5:14b') -> list[dict]:
    """Extrae entidades de un post usando NER zero-shot con Ollama."""
    try:
        response = ollama.generate(
            model=model,
            system=NER_SYSTEM,
            prompt=str(text)[:1500],
            format='json',
            options={'temperature': 0},
        )
        result = json.loads(response['response'])
        return result if isinstance(result, list) else []
    except Exception:
        return []

def run_ner_on_sample(sample_df: pd.DataFrame, text_col: str, label: str) -> pd.DataFrame:
    """Corre NER sobre una muestra y devuelve DataFrame de entidades."""
    from tqdm.auto import tqdm
    records = []
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=label):
        entities = extract_entities(row[text_col])
        for ent in entities:
            records.append({
                'sample': label,
                'entity': ent.get('entity', '').strip(),
                'type': ent.get('type', 'UNKNOWN'),
                'userid': row.get('userid', ''),
            })
    return pd.DataFrame(records)

print("Función de NER lista.")
print("Verificar que qwen2.5:14b esté disponible:")
!ollama list | grep qwen2.5

In [ ]:
# Ejecutar NER en las tres muestras
# Carga desde caché si existe, si no computa y guarda
NER_CACHE = IM_RESULTS / 'ner_comparison.parquet'

if NER_CACHE.exists():
    ner_all = pd.read_parquet(NER_CACHE)
    print(f"NER cargado desde caché: {len(ner_all):,} entidades")
else:
    print("Ejecutando NER en las tres muestras (puede tardar 30-60 min)...")
    text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
    
    ner_parts = []
    for sample_df, label in [
        (sample_random,     'aleatoria'),
        (sample_stratified, 'estratificada'),
        (sample_top_users,  'top_usuarios'),
    ]:
        df_ner = run_ner_on_sample(sample_df, text_col, label)
        ner_parts.append(df_ner)
    
    ner_all = pd.concat(ner_parts, ignore_index=True)
    ner_all = ner_all[ner_all['entity'].str.len() > 1]  # eliminar entidades vacías
    ner_all.to_parquet(NER_CACHE, index=False)
    print(f"NER guardado: {len(ner_all):,} entidades")

In [ ]:
# Comparativa de muestras NER — plotly subplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(ner_all) > 0:
    sample_labels = ['aleatoria', 'estratificada', 'top_usuarios']
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f'Top entidades<br>({s})' for s in sample_labels],
        horizontal_spacing=0.12,
    )

    for col_idx, s_label in enumerate(sample_labels, 1):
        subset = ner_all[ner_all['sample'] == s_label]
        top_ents = subset['entity'].value_counts().head(15)
        fig.add_trace(
            go.Bar(
                x=top_ents.values[::-1],
                y=top_ents.index[::-1],
                orientation='h',
                marker_color='#E94E4E',
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        fig.update_xaxes(title_text='menciones', row=1, col=col_idx)

    fig.update_layout(
        title='NER — comparativa de estrategias de muestreo',
        template='plotly_dark',
        height=550, width=1100,
        margin=dict(l=140, r=20, t=80, b=40),
    )
    fig.show()

    # Estabilidad: intersección de top-20 entre estrategias
    top_per_sample = {
        s: set(ner_all[ner_all['sample'] == s]['entity'].value_counts().head(20).index)
        for s in sample_labels
    }
    todas = top_per_sample['aleatoria'] & top_per_sample['estratificada'] & top_per_sample['top_usuarios']
    print(f"Entidades en top-20 de las 3 estrategias ({len(todas)}): {sorted(todas)}")
else:
    print("Sin datos NER disponibles.")

In [ ]:
# Top entidades por tipo — plotly subplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(ner_all) > 0:
    entity_types = [t for t in ['PERSON', 'ORGANIZATION', 'IDEOLOGY', 'EVENT', 'LOCATION']
                    if len(ner_all[ner_all['type'] == t]) > 0]
    
    fig = make_subplots(
        rows=1, cols=len(entity_types),
        subplot_titles=entity_types,
        horizontal_spacing=0.08,
    )
    colors = {'PERSON': '#E94E4E', 'ORGANIZATION': '#4E9EE9', 'IDEOLOGY': '#E9A84E',
              'EVENT': '#7AE94E', 'LOCATION': '#C44EE9'}

    for col_idx, etype in enumerate(entity_types, 1):
        subset = ner_all[ner_all['type'] == etype]
        top = subset['entity'].value_counts().head(10)
        fig.add_trace(
            go.Bar(
                x=top.values[::-1],
                y=top.index[::-1],
                orientation='h',
                marker_color=colors.get(etype, '#E94E4E'),
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        fig.update_xaxes(title_text='menciones', row=1, col=col_idx)

    fig.update_layout(
        title='Top 10 entidades por tipo (todas las muestras combinadas)',
        template='plotly_dark',
        height=420, width=230 * len(entity_types),
        margin=dict(l=120, r=20, t=60, b=40),
    )
    fig.show()
else:
    print("Sin datos NER disponibles.")

In [ ]:
# Red de co-ocurrencia de entidades — ¿qué entidades aparecen juntas en el mismo post?
# Revela asociaciones ideológicas y narrativas recurrentes en el foro
import plotly.graph_objects as go
import networkx as nx
from itertools import combinations

if len(ner_all) > 0 and 'userid' in ner_all.columns:
    # Reconstruir co-ocurrencias por post (agrupamos por userid como proxy si no hay post_id)
    # Filtramos a PERSON e IDEOLOGY que son las más reveladoras en un foro extremista
    ner_focus = ner_all[ner_all['type'].isin(['PERSON', 'IDEOLOGY', 'ORGANIZATION'])].copy()
    
    # Agrupar entidades por (muestra, userid) como unidad de co-ocurrencia
    grouped = ner_focus.groupby(['sample', 'userid'])['entity'].apply(list).reset_index()
    
    cooc = Counter()
    for _, row in grouped.iterrows():
        ents = list(set(row['entity']))  # deduplicar dentro del mismo "documento"
        for a, b in combinations(sorted(ents), 2):
            cooc[(a, b)] += 1
    
    # Filtrar pares poco frecuentes
    min_cooc = 2
    cooc_filtered = {k: v for k, v in cooc.items() if v >= min_cooc}
    
    if cooc_filtered:
        G_ner = nx.Graph()
        for (a, b), w in cooc_filtered.items():
            G_ner.add_edge(a, b, weight=w)
        
        # Top 40 nodos por degree
        top_nodes = sorted(G_ner.degree(), key=lambda x: x[1], reverse=True)[:40]
        top_set = {n for n, _ in top_nodes}
        G_viz = G_ner.subgraph(top_set).copy()
        
        pos = nx.spring_layout(G_viz, k=1.5, iterations=50, seed=42)
        
        edge_x, edge_y, edge_w = [], [], []
        for u, v, d in G_viz.edges(data=True):
            x0, y0 = pos[u]; x1, y1 = pos[v]
            edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
            edge_w.append(d.get('weight', 1))
        
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y, mode='lines',
            line=dict(width=0.8, color='rgba(150,150,150,0.3)'),
            hoverinfo='none',
        )
        
        node_deg = [G_viz.degree(n) for n in G_viz.nodes()]
        node_trace = go.Scatter(
            x=[pos[n][0] for n in G_viz.nodes()],
            y=[pos[n][1] for n in G_viz.nodes()],
            mode='markers+text',
            marker=dict(
                size=[d * 4 + 8 for d in node_deg],
                color=node_deg,
                colorscale='Plasma',
                showscale=True,
                colorbar=dict(title='Co-ocurrencias'),
            ),
            text=list(G_viz.nodes()),
            textposition='top center',
            textfont=dict(size=8, color='white'),
            hoverinfo='text',
        )
        
        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                title='Red de co-ocurrencia de entidades — IronMarch (PERSON, IDEOLOGY, ORGANIZATION)',
                template='plotly_dark',
                showlegend=False,
                width=950, height=650,
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                margin=dict(l=20, r=20, t=60, b=20),
            )
        )
        fig.show()
        print(f"Red NER: {G_viz.number_of_nodes()} entidades, {G_viz.number_of_edges()} pares co-ocurrentes")
    else:
        print("Sin co-ocurrencias suficientes (min_cooc={min_cooc}). Ejecutar NER primero.")
else:
    print("Sin datos NER disponibles.")

<a id='s8'></a>
## Sección 8 — Estilometría y atribución

Comparación estilométrica entre usuarios. Alta similitud entre dos usuarios distintos puede indicar:
- Misma persona con dos cuentas
- Influencia directa de un usuario sobre otro (imitación de estilo)
- Ghostwriting

In [ ]:
# Preparar corpus estilométrico — top 100 usuarios con ≥10 posts
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'

stylo_df = posts[['userid', text_col]].copy()
stylo_df = stylo_df.dropna(subset=[text_col]).rename(columns={text_col: 'text'})
stylo_df = stylo_df[stylo_df['text'].astype(str).str.strip().str.len() > 20]

user_counts = stylo_df.groupby('userid').size()
active_users = user_counts[user_counts >= 10].index
stylo_df = stylo_df[stylo_df['userid'].isin(active_users)]

top_100 = user_counts[user_counts >= 10].nlargest(100).index
stylo_sample = stylo_df[stylo_df['userid'].isin(top_100)]

print(f"Usuarios con ≥10 posts:  {len(active_users):,}")
print(f"Usuarios en muestra top-100: {stylo_sample['userid'].nunique():,}")
print(f"Posts en muestra:            {len(stylo_sample):,}")

In [ ]:
# Calcular matriz de similitud estilométrica — heatmap plotly interactivo
import plotly.graph_objects as go

if len(stylo_sample) > 0 and stylo_sample['userid'].nunique() >= 2:
    sim_matrix = compare_users(stylo_sample, user_col='userid', text_col='text')
    print(f"Matriz de similitud: {sim_matrix.shape[0]}×{sim_matrix.shape[1]}")

    # Top 30 usuarios por volumen de posts
    top_30 = user_counts[user_counts >= 10].nlargest(30).index
    top_30 = [u for u in top_30 if u in sim_matrix.index]
    sim_sub = sim_matrix.loc[top_30, top_30]
    labels = [uid_to_name.get(u, str(u))[:18] for u in sim_sub.index]

    fig = go.Figure(go.Heatmap(
        z=sim_sub.values,
        x=labels,
        y=labels,
        colorscale='YlOrRd',
        zmin=0, zmax=1,
        hovertemplate='%{y} vs %{x}<br>similitud=%{z:.3f}<extra></extra>',
    ))
    fig.update_layout(
        title='Similitud estilométrica — top 30 usuarios IronMarch (hover para valores)',
        template='plotly_dark',
        height=700, width=750,
        xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9)),
        margin=dict(l=130, r=20, t=60, b=130),
    )
    fig.show()
else:
    print("Muestra insuficiente para cálculo estilométrico.")

In [ ]:
# Pares con similitud alta — candidatos a identidades duplicadas o atribución
if 'sim_matrix' in dir() and len(sim_matrix) > 0:
    users_list = sim_matrix.index.tolist()
    pairs = [
        {
            'user_a': uid_to_name.get(users_list[i], str(users_list[i])),
            'user_b': uid_to_name.get(users_list[j], str(users_list[j])),
            'similitud': round(float(sim_matrix.iloc[i, j]), 4),
        }
        for i in range(len(users_list))
        for j in range(i + 1, len(users_list))
        if sim_matrix.iloc[i, j] > 0.90
    ]
    
    pairs_df = pd.DataFrame(pairs).sort_values('similitud', ascending=False)
    print(f"Pares con similitud > 0.90: {len(pairs_df)}")
    if len(pairs_df) > 0:
        print("\n→ Candidatos para atribución de autoría o cuentas duplicadas:")
        print(pairs_df.head(20).to_string(index=False))
else:
    print("Ejecutar primero la celda de cálculo de similitud.")

In [ ]:
# Red de similitud estilométrica — grafo de usuarios con similitud > umbral
# Los clústeres en este grafo son candidatos a cuentas duplicadas o influencia directa
import plotly.graph_objects as go
import networkx as nx

STYLO_THRESHOLD = 0.75  # ajustar según distribución

if 'sim_matrix' in dir() and len(sim_matrix) > 0:
    G_stylo = nx.Graph()
    users_list = sim_matrix.index.tolist()

    for i in range(len(users_list)):
        for j in range(i + 1, len(users_list)):
            sim = float(sim_matrix.iloc[i, j])
            if sim >= STYLO_THRESHOLD:
                G_stylo.add_edge(users_list[i], users_list[j], weight=sim)

    print(f"Umbral: {STYLO_THRESHOLD} | Nodos: {G_stylo.number_of_nodes()} | Aristas: {G_stylo.number_of_edges()}")

    if G_stylo.number_of_edges() > 0:
        pos = nx.spring_layout(G_stylo, k=2.0, iterations=60, seed=42)
        
        edge_x, edge_y, edge_hover = [], [], []
        for u, v, d in G_stylo.edges(data=True):
            x0, y0 = pos[u]; x1, y1 = pos[v]
            edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
        
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y, mode='lines',
            line=dict(width=1.5, color='rgba(233,78,78,0.4)'),
            hoverinfo='none',
        )
        
        node_names = [uid_to_name.get(n, str(n)) for n in G_stylo.nodes()]
        node_deg   = [G_stylo.degree(n) for n in G_stylo.nodes()]

        node_trace = go.Scatter(
            x=[pos[n][0] for n in G_stylo.nodes()],
            y=[pos[n][1] for n in G_stylo.nodes()],
            mode='markers+text',
            marker=dict(
                size=[d * 8 + 10 for d in node_deg],
                color=node_deg,
                colorscale='YlOrRd',
                showscale=True,
                colorbar=dict(title='Conexiones'),
            ),
            text=node_names,
            textposition='top center',
            textfont=dict(size=9, color='white'),
            hoverinfo='text',
        )

        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                title=f'Red estilométrica IronMarch (similitud ≥ {STYLO_THRESHOLD})',
                template='plotly_dark',
                showlegend=False,
                width=900, height=620,
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                margin=dict(l=20, r=20, t=60, b=20),
            )
        )
        fig.show()
    else:
        print(f"Sin pares con similitud ≥ {STYLO_THRESHOLD}. Probar un umbral menor.")
else:
    print("Ejecutar primero la celda de cálculo de similitud.")

<a id='s9'></a>
## Sección 9 — Conclusiones

### Hallazgos

1. **Brokers de radicalización**: betweenness centrality identifica los conectores entre subcomunidades. Comparar red pública vs. privada revela si los brokers actúan igual en ambos espacios o si hay una capa oculta de coordinación.

2. **Validación del muestreo de embeddings**: la correlación de Spearman entre los tres métodos (embed_users, centroids completo, centroids muestreado) cuantifica exactamente cuánta información se pierde al muestrear. Si ρ > 0.90, el muestreo es válido para datasets más grandes.

3. **Topics vs. subforos**: el topic modeling no supervisado debería recuperar las categorías del foro. Si coincide, el método es generalizable a foros sin etiquetas de subforo.

4. **Estabilidad del NER**: la comparativa de estrategias de muestreo valida que una muestra pequeña es representativa para extracción de entidades.

### Por qué este dataset es excepcional para la clase

La existencia de **ground truth** — miembros públicamente identificados en investigaciones judiciales — permite validar el análisis computacional. Cuando el modelo coincide con la identidad conocida, aumenta la confianza en el método aplicado a casos sin ground truth.

### Limitaciones y ética

- El análisis es exploratorio: ningún resultado es conclusivo sin corroboración independiente
- Los datos contienen información de personas reales; el manejo debe ser mínimamente invasivo
- Objetivo defensivo: entender cómo operan estas redes, no publicar perfiles individuales

In [ ]:
# Resumen ejecutivo del análisis
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 60)
print("CASO 3 — IRONMARCH — RESUMEN EJECUTIVO")
print("=" * 60)

# Dataset
print(f"\n📊 DATASET")
print(f"  Usuarios:             {len(users):,}")
print(f"  Posts:                {len(posts):,}")
print(f"  Mensajes privados:    {len(pms):,}")
print(f"  Subforos:             {len(forums):,}")

# Redes
print(f"\n🕸️  REDES SOCIALES")
if G_public.number_of_nodes() > 0:
    lcc_public = max(nx.connected_components(G_public), key=len)
    print(f"  Red pública:  {G_public.number_of_nodes():,} nodos, {G_public.number_of_edges():,} aristas")
    print(f"  Componente principal: {len(lcc_public):,} usuarios ({len(lcc_public)/G_public.number_of_nodes()*100:.0f}%)")
    top3_public = [(uid_to_name.get(u, u), round(s, 4)) for u, s in top_brokers[:3]]
    print(f"  Top brokers públicos: {top3_public}")
if G_pm.number_of_nodes() > 0:
    print(f"  Red privada:  {G_pm.number_of_nodes():,} nodos, {G_pm.number_of_edges():,} aristas")
    top3_pm = [(uid_to_name.get(u, u), round(s, 4)) for u, s in top_pm_brokers[:3]]
    print(f"  Top brokers PMs:     {top3_pm}")

# Embeddings
print(f"\n🧠 EMBEDDINGS (Sección 5)")
for label, path_pat in [('A — embed_users',       's5a_embed_users_*.npz'),
                         ('B — centroids_full',     's5b_centroids_full_*.npz'),
                         ('C — centroids_sampled',  's5c_centroids_sampled_*.npz')]:
    matches = sorted(IM_RESULTS.glob(path_pat))
    status = matches[-1].name if matches else '⚠️ no generado'
    print(f"  {label}: {status}")
if spearman_results:
    for (lx, ly), rho in spearman_results.items():
        print(f"  Spearman {lx.split('(')[1].rstrip(')')} vs {ly.split('(')[1].rstrip(')')}: ρ={rho:.4f}")

# Topic modeling
print(f"\n📝 TOPIC MODELING (Sección 6)")
if 'n_topics' in dir():
    print(f"  Topics encontrados: {n_topics}")
    print(f"  Posts sin topic:    {topics.count(-1):,}")

# NER
print(f"\n🏷️  NER (Sección 7)")
NER_CACHE = IM_RESULTS / 'ner_comparison.parquet'
if NER_CACHE.exists():
    print(f"  Entidades extraídas: {len(ner_all):,}")
    top_ent = ner_all['entity'].value_counts().head(3)
    print(f"  Top entidades: {dict(top_ent)}")
else:
    print("  ⚠️ NER no ejecutado (requiere Ollama)")

# Estilometría
print(f"\n✍️  ESTILOMETRÍA (Sección 8)")
if 'sim_matrix' in dir() and len(sim_matrix) > 0:
    print(f"  Usuarios comparados: {sim_matrix.shape[0]}")
    high_sim = [(sim_matrix.index[i], sim_matrix.index[j], float(sim_matrix.iloc[i,j]))
                for i in range(len(sim_matrix))
                for j in range(i+1, len(sim_matrix))
                if sim_matrix.iloc[i,j] > 0.90]
    print(f"  Pares con similitud > 0.90: {len(high_sim)}")
else:
    print("  ⚠️ Estilometría no ejecutada")

print("=" * 60)